In [3]:
import numpy as np
import matplotlib.pyplot as plt
import yapss
from yapss import math
import time as timing

problem = yapss.Problem(name="Simple_Cycler", nx=[6], nd = 3)
mu = 1.2150584270572e-2

file_name = "2dorbit"
path_modifier = "../Standard_Cycler_Generation/"
reference_name = path_modifier + file_name + ".npz"
path_modifier2 = "../Inclined_Cycler_Data/"
export_name = path_modifier2 + "tgt_"

In [ ]:
C_value = 3.151175879508174
W_v = 1
W_th = 1
step = 1
initial = 1
for angle in range(initial, 91, step):
    start = timing.time()
    theta = np.radians(angle)
    phi = np.radians(step)/2

    def objective(arg):
        xi, yi, zi, vxi, vyi, vzi = arg.phase[0].initial_state
        xf, yf, zf, vxf, vyf, vzf = arg.phase[0].final_state
        tgt = [0, math.cos(-theta), math.sin(-theta)]
        c1, c2, c3 = math.cross([vxi, vyi, vzi], [vxf, vyf, vzf])
        t1, t2, t3 = math.cross([vxi, vyi, vzi], tgt)

        r1 = math.sqrt((xf + mu)**2 + yf**2 + zf**2)
        r2 = math.sqrt((xf - 1 + mu)**2 + yf**2 + zf**2)
        U = -0.5*(xf**2 + yf**2) - (1 - mu)/r1 - mu/r2

        Cf = -2*U - (vxf**2 + vyf**2 + vzf**2)
        arg.objective = W_v*(c1**2 + c2**2 + c3**2) + W_th*(t1**2 + t2**2 + t3**2)

    def continuous(arg):
        x, y, z, vx, vy, vz = arg.phase[0].state

        r1 = math.sqrt((x + mu)**2 + y**2 + z**2)
        r2 = math.sqrt((x - 1 + mu)**2 + y**2 + z**2)
        U = -0.5*(x**2 + y**2) - (1 - mu)/r1 - mu/r2

        ax = x - (1 - mu)*(x + mu)/r1**3 - mu*(x - 1 + mu)/r2**3 + 2*vy
        ay = y - (1 - mu)*y/r1**3 - mu*y/r2**3 - 2*vx
        az = -(1 - mu)*z/r1**3 - mu*z/r2**3

        C = -2*U - (vx**2 + vy**2 + vz**2)

        arg.phase[0].dynamics = [vx, vy, vz, ax, ay, az]

    def discrete(arg):
        xi, yi, zi, vxi, vyi, vzi = arg.phase[0].initial_state
        xf, yf, zf, vxf, vyf, vzf = arg.phase[0].final_state
        c1, c2, c3 = math.cross([vxi, vyi, vzi], [vxf, vyf, vzf])
        arg.discrete = [xf-xi, yf-yi, zf-zi]

    problem.functions.objective = objective
    problem.functions.continuous = continuous
    problem.functions.discrete = discrete

    def day_to_gen(t):
        return t*2*math.pi/27.321661
    def gen_to_day(t):
        return t*27.321661/2/math.pi

    bounds = problem.bounds.phase[0]
    bounds.initial_time.lower = bounds.initial_time.upper = 0
    problem.bounds.discrete.lower[:] = problem.bounds.discrete.upper[:] = [0, 0, 0]

    load_tgt = export_name + f"{angle-step}.npz"
    if angle == initial:
        load_tgt = reference_name

    guess_sol = np.load(load_tgt)
    state_guess = guess_sol["state"]
    if angle == initial:
        state_guess_3d = np.zeros(tuple(x + y for x, y in zip(np.shape(state_guess), (2,0))))
        for i in range (np.size(state_guess, 1)):
            xs, ys, vxs, vys = state_guess[:,i]
            xsf = xs
            ysf = ys - 2*ys*math.sin(phi)**2
            zsf = 2*ys*math.sin(phi)*math.cos(phi)
            vxsf = vxs
            vysf = vys - 2*vys*math.sin(phi)**2
            vzsf = 2*vys*math.sin(phi)*math.cos(phi)
            state_guess_3d[:,i] = [xsf, ysf, zsf, vxsf, vysf, vzsf]
    else:
        state_guess_3d = np.zeros(np.shape(state_guess))
        for i in range (np.size(state_guess, 1)):
            xs, ys, zs, vxs, vys, vzs = state_guess[:,i]
            xsf = xs
            ysf = ys - 2*ys*math.sin(phi)**2
            zsf = 2*ys*math.sin(phi)*math.cos(phi)
            vxsf = vxs
            vysf = vys - 2*vys*math.sin(phi)**2
            vzsf = 2*vys*math.sin(phi)*math.cos(phi)
            state_guess_3d[:,i] = [xsf, ysf, zsf, vxsf, vysf, vzsf]

    time_guess = guess_sol["time"]

    phase = problem.guess.phase[0]
    phase.time = time_guess
    phase.state = state_guess_3d

    problem.derivatives.method = "auto"
    problem.derivatives.order = "second"
    problem.spectral_method = "lgl"
    segments, points = 100, 10
    problem.mesh.phase[0].collocation_points = segments*[points]
    problem.mesh.phase[0].fraction = segments*[1/segments]

    problem.ipopt_options.mu_strategy = "adaptive"
    problem.ipopt_options.print_level = 3
    problem.ipopt_options.print_user_options = "yes"
    problem.ipopt_options.timing_statistics = "yes"
    problem.ipopt_options.sb = "yes"
    problem.ipopt_options.tol = 1e-8

    solution = problem.solve()

    state = solution.phase[0].state
    time = solution.phase[0].time
    control = solution.phase[0].control
    time_c = solution.phase[0].time_c
    t0 = solution.phase[0].initial_time
    tf = solution.phase[0].final_time
    x, y, z, vx, vy, vz = state

    solve_time = timing.time() - start

    print("\n Degree Target", angle)
    print("Solve Time", solve_time)
    print("Orbit Period (nondimensional time)", tf)
    print("Difference in final x position", x[0] - x[-1])
    print("Difference in final y position", y[0] - y[-1])
    print("Difference in final z position", z[0] - z[-1])
    print("Difference in final x velocity", vx[0] - vx[-1])
    print("Difference in final y velocity", vy[0] - vy[-1])
    print("Difference in final z velocity", vz[0] - vz[-1])
    print("Velocity Cross Product", math.cross([vx[0], vy[0], vz[0]], [vx[-1], vy[-1], vz[-1]]))
    def U(x, y, z):
        r1 = math.sqrt((x + mu)**2 + y**2 + z**2)
        r2 = math.sqrt((x - 1 + mu)**2 + y**2 + z**2)
        return -0.5*(x**2 + y**2) - (1 - mu)/r1 - mu/r2

    C_vals = np.zeros((len(x), 1))
    for i in range (len(x)):
        C_vals[i] = -2*U(x[i], y[i], z[i]) - (vx[i]**2 + vy[i]**2 + vz[i]**2)
    C_err = np.ptp(C_vals) # Maximum Jacobi constant deviation
    print("Maximum Jacobi Constant Deviation", C_err)
    hamiltonian = solution.phase[0].hamiltonian
    hamil_dev = np.ptp(hamiltonian)
    print("Maximum Hamiltonian Deviation", hamil_dev)
    print("Angle", math.rad2deg(math.arccos(math.dot([vy[0], vz[0]], [-1, 0])/math.sqrt(vy[0]**2 + vz[0]**2))))

    name = export_name + f"{angle}"
    np.savez(
        name,
        state = state,
        time = time,
        C_vals = C_vals,
        hamiltonian = hamiltonian
    )


 Degree Target 1
Solve Time 4.885072231292725
Orbit Period (nondimensional time) 9.800403388291363
Difference in final x position 0.0
Difference in final y position 0.0
Difference in final z position 0.0
Difference in final x velocity -1.1496725357897556e-11
Difference in final y velocity 9.492406860545088e-15
Difference in final z velocity -3.6578465950620753e-13
Velocity Cross Product [-7.50001190e-14  4.11650689e-14  2.35834522e-12]
Maximum Jacobi Constant Deviation 1.0480594170303448e-10
Maximum Hamiltonian Deviation 5.88171536547577e-16
Angle 1.0000000000090123

 Degree Target 2
Solve Time 4.516289472579956
Orbit Period (nondimensional time) 9.800737170960119
Difference in final x position 0.0
Difference in final y position 0.0
Difference in final z position 0.0
Difference in final x velocity -1.0264813224687359e-11
Difference in final y velocity 8.745781876484671e-14
Difference in final z velocity -4.357668739740639e-12
Velocity Cross Product [-8.93197192e-13  7.35245685e-14  2.